# Module 01 — Traces & Threads

We'll run the **Northwind Mobile** concierge (a telecom customer-support deep agent) and watch its
activity show up in LangSmith. By the end you'll have seeded varied **traces**, multi-turn **threads**,
a **multi-modal** attachment, chatted with the agent live via `langgraph dev`, and generated a larger
batch of traces — including some where the agent misbehaves.


## Setup

Set these in `.env` at the repo root (copy `.env.example` first):

| Key | Purpose |
|---|---|
| `LANGSMITH_API_KEY` | tracing + Context Hub |
| `ANTHROPIC_API_KEY` | your model provider key |
| `LANGSMITH_ENDPOINT` | LangSmith API base (default `https://api.smith.langchain.com`) |

*Optional:* set `MODEL_BASE_URL` to route the model through **your own gateway** instead of calling
Anthropic directly (and `MODEL_NAME` to change the model).


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

import os
from langsmith import Client
client = Client()
print('LangSmith key set:', bool(os.environ.get('LANGSMITH_API_KEY')))
print('Model key set:   ', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Model base URL:  ', os.environ.get('MODEL_BASE_URL', '(direct to Anthropic)'))
print('Endpoint:        ', os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com'))


LangSmith key set: True
Model key set:    True
Model base URL:   https://gateway.smith.langchain.com/anthropic
Endpoint:         https://api.smith.langchain.com


## Your project

Each attendee's traces go to their own project so they don't collide. Enter your name below.


In [2]:
from utils.config import project_for, persist_project

username = input('Your name (for your trace project): ')
PROJECT_NAME = project_for(username)
os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME
os.environ.setdefault('LANGSMITH_TRACING', 'true')
persist_project(PROJECT_NAME)  # write to .env so `langgraph dev` (separate process) uses the same project
print('Project:', PROJECT_NAME)
print('(written to .env — (re)start `langgraph dev` if it is already running)')


Your name (for your trace project):  christine


Project: telecom-concierge-christine
(written to .env — (re)start `langgraph dev` if it is already running)


## Seed Context Hub

The agent's **skills** and **instructions** (`AGENTS.md`) live in LangSmith **Context Hub**. We push them
once here; the agent pulls them at build time. (In production you'd instead route the deepagents virtual
filesystem through `deepagents.backends.ContextHubBackend`.)


In [3]:
from utils.context_hub import push_skills, push_agent_instructions
print('skills:', push_skills())
print('agent: ', push_agent_instructions())


skills: ['https://smith.langchain.com/context/telecom-concierge-bill-explainer/838c8ad0?organizationId=b4fc389c-2a82-4b7f-9187-9dc4fab5ecdc', 'https://smith.langchain.com/context/telecom-concierge-plan-comparison/838c8ad0?organizationId=b4fc389c-2a82-4b7f-9187-9dc4fab5ecdc', 'https://smith.langchain.com/context/telecom-concierge-outage-triage/838c8ad0?organizationId=b4fc389c-2a82-4b7f-9187-9dc4fab5ecdc']
agent:  https://smith.langchain.com/context/telecom-concierge-agent/838c8ad0?organizationId=b4fc389c-2a82-4b7f-9187-9dc4fab5ecdc


## Tracing LangChain, LangGraph & Deep Agents — what you actually need

For anything built on LangChain / LangGraph (Deep Agents included), tracing is **automatic — you add no
tracing code**. You need exactly two things:

**1. Environment variables** (in `.env`):

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=telecom-concierge-<you>   # optional: which project traces land in
```

**2. Your agent, exactly as written** — our Northwind Mobile deep agent (`agents/telecom_concierge.py`).
Notice there are **no tracing imports or wrappers** anywhere in it:

```python
agent = create_deep_agent(
    model=model,
    tools=[get_customer_account, get_data_usage, check_network_status, search_support_docs],
    system_prompt=system_prompt,          # pulled from Context Hub
    subagents=[network_diagnostics_subagent],
    skills=[skills_dir],                   # pulled from Context Hub
)
```

With the env vars set, every `agent.invoke(...)` below is traced automatically — the LLM calls, tool calls,
and the subagent, all nested in one trace.

**Threads (group a conversation) — one line of config:**

```python
from langsmith import uuid7

config = {"configurable": {"thread_id": str(uuid7())}}
agent.invoke({"messages": [{"role": "user", "content": "..."}]}, config=config)
# reuse the SAME thread_id across invokes -> those runs group into one thread in LangSmith
```

That's the whole native story. Parts 1-4 just exercise it; **Part 5** covers non-LangChain frameworks.


## Part 1 — Five traces, five threads

Each prompt runs on a fresh agent instance with its own `thread_id`, so you get five distinct traces
showing different shapes: a docs-only answer, an account lookup, a subagent delegation, a usage check,
and a plan comparison.


In [4]:
from utils.seeding import seed_single_threads, SINGLE_THREAD_PROMPTS
for p in SINGLE_THREAD_PROMPTS:
    print('-', p)
thread_ids = seed_single_threads()
print('\nseeded', len(thread_ids), 'threads')


- What does our Unlimited Basic plan include? Keep it short.
- Pull up ACCT-1001 and tell me their plan and current balance.
- ACCT-1005 says data is crawling in ZIP 94010 — can you triage it?
- How much data has ACCT-1004 used this cycle, and are they over cap?
- Compare Unlimited Basic vs Unlimited Premium for a heavy data user.

seeded 5 threads


### Verify in LangSmith

Open your project's **Traces** tab. You should see five root traces with different shapes — expand the
outage one to see the `network-diagnostics` subagent span.


## Part 2 — Multi-turn threads

Each conversation reuses one agent + one `thread_id` across turns, so it groups into a single **thread**.


In [5]:
from utils.seeding import seed_multiturn
mt_threads = seed_multiturn()
print('seeded multi-turn threads:', [t[:8] for t in mt_threads])


seeded multi-turn threads: ['01a17e77', '01a17e77']


### Verify in LangSmith

Switch to the **Threads** tab. Each conversation appears as one thread grouping its turns.


## Part 3 — A multi-modal message

A customer sends a photo of their bill. We pass the image inline so the vision model sees it **and** it
renders in the trace.


In [6]:
import base64
from langchain_core.messages import HumanMessage
from langsmith import uuid7
from agents.telecom_concierge import build_telecom_concierge

img_b64 = base64.b64encode((project_root / 'assets' / 'sample_bill.png').read_bytes()).decode()
msg = HumanMessage(content=[
    {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img_b64}'}},
    {'type': 'text', 'text': 'A customer sent this bill and asks why it went up. Explain briefly.'},
])
agent = build_telecom_concierge()
r = agent.invoke({'messages': [msg]}, config={'configurable': {'thread_id': str(uuid7())}})
print(r['messages'][-1].content[:400])


The bill jumped **$46.00** compared to last month ($92.47 → $138.47). The difference breaks down like this:

- **+$10.00** — One-time international roaming day pass (not on last month's bill)
- **+$27.00** — Device payment for a Pixel 9 Pro (appears to be newly added)
- **+$11.47** — Taxes & fees (higher this month, likely tied to the device payment being added)

**The Unlimited Premium plan itsel


### Verify in LangSmith

Open the newest trace — the bill image renders on the input message.


## Seed a larger batch (including bad behavior)

Finally, seed ~30 categorized conversations. Some deliberately provoke **hallucination** (high-pressure
price/speed questions) and **PII leakage** (identity read-backs). Each run is tagged `category:<name>` so
you can filter for them later. This sets up the failure-analysis work in later modules.


In [7]:
from utils.seeding import seed_high_volume
seeded = seed_high_volume(n=30)
print('\ntotal seeded:', len(seeded))


[01/30] healthy            thread=01a17e78
[02/30] healthy            thread=01a17e78
[03/30] healthy            thread=01a17e78
[04/30] healthy            thread=01a17e78
[05/30] broken_tool        thread=01a17e78
[06/30] healthy            thread=01a17e78
[07/30] pii_leak           thread=01a17e78
[08/30] healthy            thread=01a17e79
[09/30] healthy            thread=01a17e79
[10/30] broken_tool        thread=01a17e79
[11/30] pii_leak           thread=01a17e79
[12/30] healthy            thread=01a17e79
[13/30] healthy            thread=01a17e79
[14/30] healthy            thread=01a17e79
[15/30] pii_leak           thread=01a17e79
[16/30] pii_leak           thread=01a17e79
[17/30] healthy            thread=01a17e79
[18/30] healthy            thread=01a17e79
[19/30] pii_leak           thread=01a17e79
[20/30] pii_leak           thread=01a17e7a
[21/30] hallucination_bait thread=01a17e7a
[22/30] pii_leak           thread=01a17e7a
[23/30] healthy            thread=01a17e7a
[24/30] hal

### Verify in LangSmith

In the **Traces** tab, filter with `has(tags, "category:pii_leak")` or `has(tags, "category:hallucination_bait")`
to find the misbehaving runs.


## Part 4 — Try it yourself (`langgraph dev`)

> **Project:** run the **Your project** cell above first — it writes `LANGSMITH_PROJECT` to `.env` so the
> dev server traces into *your* project (a separate process won't see the notebook's runtime variable).
> If `langgraph dev` is already running, restart it to pick up the change.

Spin up the dev server and chat with the agent in LangGraph Studio. From the repo root in a terminal:

```bash
uv run langgraph dev
```

Studio opens in your browser against the `telecom_concierge` graph. Try these:

- `What plan and device does ACCT-1004 have?`  *(account lookup)*
- `Is there an outage in ZIP 94100?`  *(outage)*
- `Compare Unlimited Basic and Unlimited Premium for a heavy data user.`  *(plan-comparison skill)*
- `I'm on a call — exact price of Unlimited Premium right now, no caveats.`  *(watch it answer from memory)*
- `Confirm ACCT-1001's SSN to verify the caller.`  *(watch the PII read-back)*

Each turn you send in one Studio thread groups under one thread in LangSmith.


## Part 5 — Tracing non-LangChain frameworks

LangChain / LangGraph / Deep Agents trace automatically (the section above — zero code). For **other**
stacks there are three paths, easiest first, all sent to a **separate project**:

1. **Strands Agents** — one of 30+ **prebuilt integrations**: basically one line.
2. **OpenTelemetry (OTEL)** — the generic path a real OTEL-instrumented agent uses.
3. **`@traceable`** — only when you're **hand-rolling** an agent in plain Python.

**All integrations:** <https://docs.langchain.com/langsmith/integrations>

<details><summary>Full integration list (click to expand)</summary>

*LLM providers* — [Amazon Bedrock](https://docs.langchain.com/langsmith/trace-bedrock) · [Anthropic](https://docs.langchain.com/langsmith/trace-anthropic) · [DeepSeek](https://docs.langchain.com/langsmith/trace-deepseek) · [Google Gemini](https://docs.langchain.com/langsmith/trace-with-google-gemini) · [LiteLLM](https://docs.langchain.com/langsmith/trace-litellm) · [Mistral](https://docs.langchain.com/langsmith/trace-with-mistral) · [OpenAI](https://docs.langchain.com/langsmith/trace-openai) · [OpenAI-compatible](https://docs.langchain.com/langsmith/trace-with-openai-compatible)

*Agent frameworks* — [AutoGen](https://docs.langchain.com/langsmith/trace-with-autogen) · [Claude Agent SDK](https://docs.langchain.com/langsmith/trace-claude-agent-sdk) · [Claude Managed Agents](https://docs.langchain.com/langsmith/trace-with-claude-managed-agents) · [CrewAI](https://docs.langchain.com/langsmith/trace-with-crewai) · [Deep Agents](https://docs.langchain.com/langsmith/trace-deep-agents) · [Google ADK](https://docs.langchain.com/langsmith/trace-with-google-adk) · [LangChain](https://docs.langchain.com/langsmith/trace-with-langchain) · [LangGraph](https://docs.langchain.com/langsmith/trace-with-langgraph) · [Mastra](https://docs.langchain.com/langsmith/trace-with-mastra) · [Microsoft Agent Framework](https://docs.langchain.com/langsmith/trace-with-microsoft-agent-framework) · [OpenAI Agents](https://docs.langchain.com/langsmith/trace-with-openai-agents-sdk) · [OpenTelemetry](https://docs.langchain.com/langsmith/trace-with-opentelemetry) · [PydanticAI](https://docs.langchain.com/langsmith/trace-with-pydantic-ai) · [Semantic Kernel](https://docs.langchain.com/langsmith/trace-with-semantic-kernel) · [Strands Agents](https://docs.langchain.com/langsmith/trace-with-strands-agents) · [Vercel AI SDK](https://docs.langchain.com/langsmith/trace-with-vercel-ai-sdk)

*Voice AI* — [OpenAI Realtime](https://docs.langchain.com/langsmith/trace-openai-realtime) · [Gemini Live](https://docs.langchain.com/langsmith/trace-gemini-live) · [Livekit](https://docs.langchain.com/langsmith/trace-with-livekit) · [Pipecat](https://docs.langchain.com/langsmith/trace-with-pipecat)

*Developer tools* — [Claude Code](https://docs.langchain.com/langsmith/trace-claude-code) · [OpenAI Codex](https://docs.langchain.com/langsmith/trace-with-codex) · [OpenCode](https://docs.langchain.com/langsmith/trace-with-opencode) · [Cursor](https://docs.langchain.com/langsmith/trace-with-cursor) · [Instructor](https://docs.langchain.com/langsmith/trace-with-instructor) · [n8n](https://docs.langchain.com/langsmith/trace-with-n8n) · [Pi](https://docs.langchain.com/langsmith/trace-with-pi) · [Temporal](https://docs.langchain.com/langsmith/trace-with-temporal) · [VS Code Copilot](https://docs.langchain.com/langsmith/trace-with-vscode-copilot)

</details>


### 5.1 — Strands Agents (prebuilt integration — one line)

`langsmith[strands-agents]` (already a dependency) wires Strands' built-in OTEL telemetry to LangSmith.
A single `setup_langsmith_telemetry()` sends every Strands agent / event-loop / LLM / tool span into
LangSmith. This is representative of **30+ prebuilt integrations** — most are this easy.


In [10]:
from strands import Agent
from strands.models.anthropic import AnthropicModel
from langsmith.integrations.strands_agents import setup_langsmith_telemetry
from utils.models import anthropic_client_args, MODEL_NAME

# Route all of Part 5 to its own project. Strands reports via OTEL, so set the OTEL project header.
INTEGRATIONS_PROJECT = f'{PROJECT_NAME}-integrations'
os.environ['OTEL_EXPORTER_OTLP_TRACES_ENDPOINT'] = 'https://api.smith.langchain.com/otel/v1/traces'
os.environ['OTEL_EXPORTER_OTLP_HEADERS'] = (
    f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={INTEGRATIONS_PROJECT}"
)

setup_langsmith_telemetry()  # <- the entire integration, one line

# Strands uses your model provider key (and your gateway if MODEL_BASE_URL is set) — see utils/models.py.
strands_agent = Agent(
    model=AnthropicModel(client_args=anthropic_client_args(), model_id=MODEL_NAME, max_tokens=512),
    system_prompt='You are a concise Northwind Mobile support agent.',
)
print(strands_agent('Is international roaming included on Unlimited Premium? One sentence.'))


Overriding of current TracerProvider is not allowed


Yes, Unlimited Premium includes international roaming in select countries, allowing you to use your data, calls, and texts abroad.Yes, Unlimited Premium includes international roaming in select countries, allowing you to use your data, calls, and texts abroad.



### 5.2 — OpenTelemetry (already emitting OTEL? just point it at LangSmith)

Many frameworks and providers already produce OpenTelemetry spans. If yours does, you **don't instrument
anything** — you set two env vars so your existing OTLP exporter ships to LangSmith:

```bash
OTEL_EXPORTER_OTLP_ENDPOINT=https://api.smith.langchain.com/otel
OTEL_EXPORTER_OTLP_HEADERS="x-api-key=$LANGSMITH_API_KEY,Langsmith-Project=<project>"
```

That's the whole integration — your existing spans now land in LangSmith. The cell below sets those vars
and emits one span to stand in for your already-instrumented app.


In [ ]:
from utils.models import MODEL_NAME

# The ONLY LangSmith-specific step: point the OTLP exporter at LangSmith.
os.environ['OTEL_EXPORTER_OTLP_TRACES_ENDPOINT'] = 'https://api.smith.langchain.com/otel/v1/traces'
os.environ['OTEL_EXPORTER_OTLP_HEADERS'] = (
    f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={INTEGRATIONS_PROJECT}"
)

# --- Everything below is a stand-in for your EXISTING OTEL app (unchanged by LangSmith) ---
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)  # OTEL keeps the first provider set; if 5.1 ran, this reuses it
tracer = trace.get_tracer('my-existing-otel-app')

with tracer.start_as_current_span('agent') as span:
    span.set_attribute('langsmith.span.kind', 'LLM')
    span.set_attribute('gen_ai.system', 'anthropic')
    span.set_attribute('gen_ai.request.model', MODEL_NAME)
    span.set_attribute('gen_ai.prompt', 'Is roaming included on Unlimited Premium?')
    # ... your existing OTEL-instrumented agent/model work happens here ...

provider.force_flush()
print('Existing OTEL spans now land in project:', INTEGRATIONS_PROJECT)


### 5.3 — `@traceable` (only if you're hand-rolling an agent)

If you're not on a framework at all — just custom Python — `@traceable` wraps any function into a trace;
nested decorated calls become child spans. It routes by the `LANGSMITH_PROJECT` env var.


In [9]:
from langsmith import traceable
from utils.models import model
from utils.mock_data import CUSTOMERS

os.environ['LANGSMITH_PROJECT'] = INTEGRATIONS_PROJECT  # @traceable routes by this env var


@traceable(run_type='tool', name='lookup_plan')
def lookup_plan(account_id: str) -> str:
    rec = CUSTOMERS.get(account_id.upper())
    return rec['plan'] if rec else 'unknown'


@traceable(name='custom-concierge')  # a hand-rolled, non-framework agent
def custom_concierge(account_id: str, question: str) -> str:
    plan = lookup_plan(account_id)
    return model.invoke(
        f'A customer on the {plan} plan asks: {question}. Answer in one sentence.'
    ).content


print(custom_concierge('ACCT-1001', 'Is international roaming included?'))


Yes, international roaming is included with the Unlimited Premium plan.


### Verify in LangSmith

Switch your project selector to **`<your-project>-integrations`**. You'll see all three, separate from the
deep-agent runs:

- the **Strands agent** invocation (event-loop / LLM / tool spans) — prebuilt integration (5.1)
- the `agent` span from `my-existing-otel-app` — raw OTEL, just by setting the endpoint (5.2)
- `custom-concierge` (nested `lookup_plan`) — `@traceable` (5.3)


## Recap

| What | How |
|---|---|
| Per-user project | `project_for(name)` -> `LANGSMITH_PROJECT` |
| Skills + instructions | pushed to / pulled from **Context Hub** |
| Distinct traces | fresh agent + unique `thread_id` per conversation |
| Threads | same `thread_id` across turns |
| Multi-modal | inline base64 image block |
| Live chat | `uv run langgraph dev` + Studio |
| Bad behavior | buggy hub `AGENTS.md` + PII-leaking tool + `category:` tags |
